# System Architecture

```mermaid
graph TD
    U[User Query] --> O[Framework Orchestration Layer]

    subgraph FO[Framework Orchestration]
        LG[LangGraph State Machine]
        CA[CrewAI Sequential Agents]
        LI[LlamaIndex Workflows]
    end

    O --> LG
    O --> CA
    O --> LI

    LG --> P[Planner and Task Decomposition]
    CA --> P
    LI --> P

    P --> HR[Hybrid Retrieval Engine]

    subgraph RP[Retrieval Pipeline]
        B[BM25 Sparse Search]
        D[Dense Search over Chroma]
        F[RRF Fusion]
        A[Adjacent Chunk Expansion]
        R[CrossEncoder Reranking]
    end

    subgraph DL[Data Layer]
        CDB[(Chroma Vector Store)]
        BM[(BM25 Token Index)]
        CH[(SEC Chunk Corpus JSONL)]
    end

    HR --> B
    HR --> D
    B --> F
    D --> F
    F --> A
    A --> R
    R --> G[Generator]
    G --> Critic{Quality and Grounding Critic}

    Critic -. "REPAIR: Insufficient Context" .-> P
    Critic -. "REVISE: Hallucination Detected" .-> G
    Critic ==>|ACCEPT| Final[Verified Financial Report]

    BM -. "serves sparse index" .- B
    CDB -. "serves dense vectors" .- D
    CH -. "corpus source" .- BM
    CH -. "corpus source" .- CDB

    classDef criticNode fill:#ffe8cc,stroke:#d97904,stroke-width:2px,color:#5c3a00;
    classDef finalNode fill:#d9f2d9,stroke:#2e7d32,stroke-width:3px,color:#1b5e20;
    class Critic criticNode;
    class Final finalNode;

    linkStyle 16 stroke:#d9534f,stroke-width:2px,stroke-dasharray:6 4;
    linkStyle 17 stroke:#f0ad4e,stroke-width:2px,stroke-dasharray:6 4;
    linkStyle 18 stroke:#2e7d32,stroke-width:4px;
```

# Agentic Workflow (LangGraph Conditional Edges)

```mermaid
graph LR
    QP[Query Planner] --> HR[Hybrid Retriever]
    HR --> CE[Context Evaluator]

    CE -- Relevant --> GN[Generator]
    CE -- Not Relevant and Retries Left --> ICR[Increment Context Retry]
    ICR --> HR

    GN --> Critic{Quality and Grounding Critic}

    Critic -- REPAIR --> RP[Repair Node]
    RP --> MRR[Mark Repair Retrieval]
    MRR --> HR

    Critic -- INSUFFICIENT_DATA --> DD[Drift Detector]
    DD --> SI[Scrape and Ingest]
    SI --> HR

    Critic -. "REPAIR: Insufficient Context" .-> QP
    Critic -. "REVISE: Hallucination Detected" .-> GN
    Critic ==>|ACCEPT| Final[Verified Financial Report]

    classDef criticNode fill:#ffe8cc,stroke:#d97904,stroke-width:2px,color:#5c3a00;
    classDef finalNode fill:#d9f2d9,stroke:#2e7d32,stroke-width:3px,color:#1b5e20;
    class Critic criticNode;
    class Final finalNode;

    linkStyle 3 stroke:#f0ad4e,stroke-width:2px,stroke-dasharray:6 4;
    linkStyle 6 stroke:#d9534f,stroke-width:2px,stroke-dasharray:6 4;
    linkStyle 12 stroke:#d9534f,stroke-width:2px,stroke-dasharray:6 4;
    linkStyle 13 stroke:#f0ad4e,stroke-width:2px,stroke-dasharray:6 4;
    linkStyle 14 stroke:#2e7d32,stroke-width:4px;
```

# Technical Legend

- Green (success path): normal progression toward a grounded final answer, e.g., Context Evaluator to Generator, Critic to End when accepted.
- Red (repair path): quality correction loop when the critic detects a fixable issue, e.g., Critic to Repair and optional Repair to re-retrieval.
- Orange (retry path): robustness loop for retrieval insufficiency, e.g., Context Evaluator retry branch and drift-driven scrape and ingest retry.
- Blue (data and retrieval components): persistent stores and retrieval stages, including BM25 index, Chroma vectors, RRF fusion, adjacent expansion, and reranking.